# Amazon Product Search

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mamoo85/open-data-actor-notebooks/blob/main/notebooks/amazon-product-search.ipynb)

**What:** Amazon product data at $0.004 per product, 20% below the alternative we tested (free plan), with 41 filled fields per product in that test. Search by keyword, URL or ASIN: price, list price, rating, reviews, Prime, badges, bought-in-past-month, images and specs. No login.

**Why:** run the [`amazon-product-search`](https://apify.com/transparent_meteorite/amazon-product-search) Apify actor from Python and analyse the result as a pandas DataFrame, with no scraping code to maintain.

> **Disclosure:** I am the author of this actor. It is pay-per-result; the free plan credit on a new Apify account covers a trial run like this one (this notebook uses a very small sample input).

## 1. Setup

In [1]:
!pip install -q apify-client pandas

import os, getpass
try:
    from google.colab import userdata
    APIFY_TOKEN = userdata.get('APIFY_TOKEN')          # Colab: add it under the key icon (Secrets)
except Exception:
    APIFY_TOKEN = os.environ.get('APIFY_TOKEN') or getpass.getpass('Apify API token: ')
assert APIFY_TOKEN, 'An Apify API token is required (a free account is enough).'
print('Token loaded.')

Token loaded.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Input
A deliberately small sample so the run is cheap and quick. See the [actor page](https://apify.com/transparent_meteorite/amazon-product-search) for every option.

In [2]:
run_input = {'keywords': ['usb c cable'], 'marketplace': 'US', 'maxItems': 3, 'maxPages': 1}

## 3. Run the actor and read the dataset

In [3]:
from apify_client import ApifyClient

client = ApifyClient(APIFY_TOKEN)
run = client.actor('transparent_meteorite/amazon-product-search').call(run_input=run_input)
# apify-client v2 returns a dict, v3 a Run object; handle both
get = (lambda k, a: run[k]) if isinstance(run, dict) else (lambda k, a: getattr(run, a))
items = list(client.dataset(get('defaultDatasetId', 'default_dataset_id')).iterate_items())
print('Run status:', get('status', 'status'), '| items:', len(items))

[apify.amazon-product-search runId:jCZIVyYMqLPryeBT5] -> Status: RUNNING, Message: 


[apify.amazon-product-search runId:jCZIVyYMqLPryeBT5] -> 2026-10-13T22:17:00.769Z ACTOR: Pulling container image of build TWciwFjSUpCctmqVS from registry.


[apify.amazon-product-search runId:jCZIVyYMqLPryeBT5] -> 2026-10-13T22:17:00.777Z ACTOR: Creating container.


[apify.amazon-product-search runId:jCZIVyYMqLPryeBT5] -> 2026-10-13T22:17:00.848Z ACTOR: Starting container.


[apify.amazon-product-search runId:jCZIVyYMqLPryeBT5] -> 2026-10-13T22:17:00.850Z ACTOR: Running under "LIMITED_PERMISSIONS".


...

[apify.amazon-product-search runId:jCZIVyYMqLPryeBT5] -> 2026-10-13T22:17:14.528Z Done. Delivered 3 (charged 3); requests 17, ok 17, blocked 0, retries 0, errors 0.


[apify.amazon-product-search runId:jCZIVyYMqLPryeBT5] -> 2026-10-13T22:17:14.609Z INFO  [Status message]: Done: 3 items


[apify.amazon-product-search runId:jCZIVyYMqLPryeBT5] -> 2026-10-13T22:17:14.651Z Thanks for using Amazon Product Search. Found a bug or need a field? Open an issue on the Issues tab (https://console.apify.com/actors/iyTOWEYZhSRYKI6vE/issues) with the run URL and we will reply there.


[apify.amazon-product-search runId:jCZIVyYMqLPryeBT5] -> Status: SUCCEEDED, Message: Done: 3 items


Run status: SUCCEEDED | items: 3


## 4. Tabulate

In [4]:
import pandas as pd
pd.set_option('display.max_colwidth', 70)
pd.set_option('display.width', 200)

df = pd.DataFrame(items)

# Privacy: remove contact / personal-data columns before anything is displayed
DROP = ["email", "phone", "author", "registeredagent", "contractor", "officer", "linkedin", "facebook", "instagram", "youtube", "tiktok", "socials", "twitter"]
KEEP = ["emailCount", "phoneCount", "emailProvider"]
df = df[[c for c in df.columns if c in KEEP or not any(p in c.lower() for p in DROP)]]
SHOW = [c for c in ["asin", "title", "brand", "price", "rating", "reviewCount", "isPrime", "boughtInPastMonth"] if c in df.columns]
df[SHOW]

,asin,title,brand,price,rating,reviewCount,isPrime,boughtInPastMonth
0,B088NRLMPV,"Anker USB C to USB C Cable, 60W Fast Charging Cable (2-Pack, 6 ft,...",Anker,9.99,4.7,89693.0,False,10K+ bought in past month
1,B0CG1LGWR6,"LISEN USB C to USB C Cable, 240W Fast Charging Type C Charger Cord...",LISEN,8.99,4.5,12505.0,True,10K+ bought in past month
2,B0HFMCRZGM,240W USB C to USB C Fast Charger Cable with Built-in Invisible Pho...,ANDHOT,6.99,NaN,NaN,False,NaN


## 5. A quick analysis

In [5]:
df['price']=pd.to_numeric(df['price'],errors='coerce')
print(df.groupby('brand')['price'].agg(['count','mean']).sort_values('count',ascending=False).head(10))
print('Top by review count:')
df.sort_values('reviewCount',ascending=False)[['title','reviewCount','rating']].head(5)

        count  mean
brand              
ANDHOT      1  6.99
Anker       1  9.99
LISEN       1  8.99
Top by review count:


,title,reviewCount,rating
0,"Anker USB C to USB C Cable, 60W Fast Charging Cable (2-Pack, 6 ft,...",89693.0,4.7
1,"LISEN USB C to USB C Cable, 240W Fast Charging Type C Charger Cord...",12505.0,4.5
2,240W USB C to USB C Fast Charger Cable with Built-in Invisible Pho...,NaN,NaN


## Next steps

- Raise the limits in the input cell to run a larger sample.
- Schedule the actor or call it from your own pipeline: [amazon-product-search on the Apify Store](https://apify.com/transparent_meteorite/amazon-product-search).
- Check the Store page for the current pay-per-result price before scaling up.

Generated by `tools/generate.py`.